# Top-k and top-p (nucleus) filtering

In [1]:
import numpy as np

logits = np.array([3.0, 2.0, 1.0, 0.0, -1.0])

def softmax(logits):
    z = np.asarray(logits, dtype=float)
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

def top_k_mask(probs, k):
    probs = np.asarray(probs, dtype=float)
    n = len(probs)
    k = int(min(max(k, 0), n))
    order = np.argsort(-probs, kind="stable")
    mask = np.zeros(n, dtype=bool)
    mask[order[:k]] = True
    return mask

def top_k_filter(probs, k):
    probs = np.asarray(probs, dtype=float)
    out = np.where(top_k_mask(probs, k), probs, 0.0)
    return out / out.sum()

def nucleus_size(probs, p_thresh):
    probs = np.asarray(probs, dtype=float)
    c = np.cumsum(np.sort(probs)[::-1])
    idx = int(np.searchsorted(c, p_thresh, side="left"))
    return int(min(idx + 1, len(probs)))

def top_p_filter(probs, p_thresh):
    probs = np.asarray(probs, dtype=float)
    n = nucleus_size(probs, p_thresh)
    order = np.argsort(-probs, kind="stable")
    out = np.zeros_like(probs)
    out[order[:n]] = probs[order[:n]]
    return out / out.sum()

In [2]:
p = softmax(logits)
print(np.round(p, 4))
print(np.round(top_k_filter(p, 2), 4))    # [0.7311 0.2689 0. 0. 0.]
print(np.round(top_p_filter(p, 0.9), 4))  # [0.6652 0.2447 0.09 0. 0.]
print(nucleus_size(p, 0.9))               # 3
print(nucleus_size(p, 0.5))               # 1
print(top_k_mask(p, 2).tolist())          # [True, True, False, False, False]
print(np.allclose(top_k_filter(p, 10), p))  # True (k >= vocab is a no-op)
print([nucleus_size(p, q) for q in (0.1, 0.5, 0.9, 0.99, 1.0)])

[0.6364 0.2341 0.0861 0.0317 0.0117]
[0.7311 0.2689 0.     0.     0.    ]
[0.6652 0.2447 0.09   0.     0.    ]
3
1
[True, True, False, False, False]
True
[1, 1, 3, 5, 5]
